# Enhancing RoomFormer with DinoV3

## 1. Overall

RoomFormer: from the paper [Connecting the Dots: Floorplan Reconstruction Using Two-Level Queries](https://github.com/ywyue/RoomFormer/tree/main#preparation)  
DINOv3: from [DINOv3](https://github.com/facebookresearch/dinov3/tree/main)  
- We want to utilize the overall architecture of RoomFormer and further improving the results on floorplan reconstruction
- In RoomFormer, 3D point cloud are projected to a density map before going to the rest of the model
- Our approach:
    - Remove the projection phase, directly feeding the point cloud to the model by leveraging DINOv3 as an intermediate layer
    - Replace current encoder with [LitePT](https://github.com/prs-eth/LitePT), keeping decoder and two-level queries intact. 

## 2. Current progress

Note: it is hard to take a large step (i.e. directly adding a 3D point cloud preprocessing layer to RoomFormer), so the work is divided to smaller steps.

### 2.1 Retraining RoomFormer

We want to reproduce the result reported in the paper of RoomFormer. The actual retrained results are shown below:

|                              | room_prec ↑ | room_rec ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | room_f1 ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------------|------------|---------------|--------------|---------------|--------------|-----------|-------------|-------------|
Results in paper               |        97.9 |       96.7 |          89.1 |         85.3 |          83.0 |         79.5 |      97.3 |        87.2 |        81.2 |
Provided model                 |        97.9 |       96.8 |          89.2 |         85.3 |          83.0 |         79.4 |      97.4 |        87.3 |        81.2 |
Retrained model                |        89.2 |       87.0 |         77.08 |         72.0 |          68.0 |         63.5 |      88.0 |        74.5 |        65.7 |
Retrained model (~1000 epochs) |        91.5 |       89.9 |          79.9 |         76.7 |          72.2 |         69.3 |      90.7 |        78.2 |        70.7 |

As can be seen, the retrained results are nowhere near the reported ones. Currently waiting for response from the author on checking if there is anything settings/hyperparameter missing.

### 2.2 Adding DINOv3 feaures on top of density map

While waiting for the reply from the author(s), it's best we move on to the next step.  

DINOv3 is involved here as an enrichment layer for the density map input of RoomFormer. Specifically, we passed the density map through RoomFormer to produce a patch token of size $(batch\_size \times embed\_dim \times 16 \times 16)$, further pass it through a linear head (consists of a convolution layer and a batch norm layer) and interpolate to get the final feature map of size $(batch\_size \times 1 \times img\_size \times img\_size)$.

![Linear Head overall architecture](./imgs/modified_rf_v1_linear_head.png "Linear Head overall architecture")

The results are reported in the table below:

|                              | room_prec ↑ | room_rec ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | room_f1 ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------------|------------|---------------|--------------|---------------|--------------|-----------|-------------|-------------|
Results in paper               |        97.9 |       96.7 |          89.1 |         85.3 |          83.0 |         79.5 |      97.3 |        87.2 |        81.2 |
Provided model                 |        97.9 |       96.8 |          89.2 |         85.3 |          83.0 |         79.4 |      97.4 |        87.3 |        81.2 |
Retrained model                |        89.2 |       87.0 |         77.08 |         72.0 |          68.0 |         63.5 |      88.0 |        74.5 |        65.7 |
Retrained model (~1000 epochs) |        91.5 |       89.9 |          79.9 |         76.7 |          72.2 |         69.3 |      90.7 |        78.2 |        70.7 |
DINOv3-enhanced model          |        88.2 |       87.2 |           7.1 |         70.5 |          68.2 |         62.5 |      87.7 |        73.7 |        65.2 |

Overall, the results are discouraging. While both the original (retrained) and the modified models perform subpar to the provided model, DINOv3-enhanced model shown little to no improvement over the retrained model.

original: 
- {'room_prec': np.float64(0.9798861581019258), 'room_rec': np.float64(0.9675093521047881), 'corner_prec': np.float64(0.8924255329792782), 'corner_rec': np.float64(0.8535414807229248), 'angles_prec': np.float64(0.8299585439624648), 'angles_rec': np.float64(0.7944150322671147), 'room_f1': np.float64(0.9736584242828102), 'corner_f1': np.float64(0.8725505177086924), 'angles_f1': np.float64(0.8117979178320776)}

retrain (500 epochs):
- {'room_prec': np.float64(0.8915358039839364), 'room_rec': np.float64(0.8696079278029484), 'corner_prec': np.float64(0.7707956450665578), 'corner_rec': np.float64(0.7199919283015213), 'angles_prec': np.float64(0.6799218778640971), 'angles_rec': np.float64(0.635443446005818), 'room_f1': np.float64(0.8804353546748717), 'corner_f1': np.float64(0.7445281309450027), 'angles_f1': np.float64(0.6569306537800096)}

retrain (1000 epochs): 
- {'room_prec': np.float64(0.9150236923058499), 'room_rec': np.float64(0.8986503411295941), 'corner_prec': np.float64(0.7987070596721617), 'corner_rec': np.float64(0.7664273130405888), 'angles_prec': np.float64(0.7219286492928096), 'angles_rec': np.float64(0.6928338815166156), 'room_f1': np.float64(0.9067631096584051), 'corner_f1': np.float64(0.7822343133261853), 'angles_f1': np.float64(0.7070820966419286)}